# Spike Pattern Analysis — All 4 Raw Cases

Standalone diagnostic notebook. Loads all 4 raw cases (complex_case1, complex_case2,
single_case1, single_case2) directly from the raw data, plots each case's 4 target
metrics over its full timeline, and quantifies step-to-step change to identify
whether any case already contains natural spikes -- a naturally-bursty case is
a stronger, non-synthetic test set for a burst-robustness evaluation than
injecting artificial spikes.

Fully self-contained: does not require any other notebook to have run first, and
does not modify `kaggle_final.ipynb` or any other existing file.


## Step 1: Platform Detection

In [ ]:
import sys
import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path

IN_COLAB  = os.path.exists('/var/colab/hostname')
IN_KAGGLE = os.path.exists('/kaggle')
if IN_COLAB:
    print("Running in Google Colab")
elif IN_KAGGLE:
    print("Running in Kaggle")
else:
    print("Running locally (not Colab, not Kaggle)")
print(f"Python version: {sys.version}")


## Step 2: Locate Raw Data Root

In [ ]:
VALID_CASE_NAMES = ['complex_case1', 'complex_case2', 'single_case1', 'single_case2']

if IN_KAGGLE:
    kaggle_input = Path('/kaggle/input')
    if not kaggle_input.exists() or not any(kaggle_input.iterdir()):
        raise FileNotFoundError(
            "/kaggle/input is empty -- attach your raw-data dataset via "
            "'Add Input' in the right sidebar before running this cell."
        )
    _candidates = sorted(
        kaggle_input.glob('**/kpi_container_cpu_usage_seconds_total.csv'),
        key=lambda p: len(p.parts)
    )
    if not _candidates:
        raise FileNotFoundError(f"No target KPI file found anywhere under {kaggle_input}.")
    raw_path = _candidates[0].parent
    while raw_path.name not in ('complex', 'single') and raw_path != raw_path.parent:
        raw_path = raw_path.parent
    if raw_path.name in ('complex', 'single'):
        raw_path = raw_path.parent
    print(f"Auto-detected raw folder at: {raw_path}")
elif IN_COLAB:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    raw_path = Path('/content/drive/My Drive/raw')
else:
    raw_path = Path('./raw')

case_dirs = {}
for case_name in VALID_CASE_NAMES:
    complexity, case_num = case_name.split('_')
    d = raw_path / complexity / case_num / 'container'
    if not d.exists():
        raise FileNotFoundError(f"{d} not found -- expected <raw_root>/{complexity}/{case_num}/container/")
    n_files = len(list(d.glob('kpi_*.csv')))
    print(f"  {case_name:16s}: {d}  ({n_files} metric files)")
    case_dirs[case_name] = d


## Step 3: Load and Pivot All 4 Cases to Wide Format

In [ ]:
TARGET_METRICS = [
    'container_cpu_usage_seconds_total',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss',
]
TARGET_NAMES = ['cpu_usage', 'mem_usage', 'mem_working_set', 'mem_rss']

def load_case(case_dir: Path, case_name: str) -> pd.DataFrame:
    csv_files = sorted(case_dir.glob('kpi_*.csv'))
    dfs = []
    for f in csv_files:
        d = pd.read_csv(f)
        if 'kpi_name' in d.columns:
            d = d[d['kpi_name'].isin(TARGET_METRICS)]
        dfs.append(d)
    combined = pd.concat(dfs, ignore_index=True)
    del dfs
    gc.collect()

    pivoted = combined.pivot_table(
        index=['timestamp', 'cmdb_id'], columns='kpi_name', values='value', aggfunc='first'
    ).reset_index()
    pivoted.columns.name = None
    del combined
    gc.collect()

    for col in TARGET_METRICS:
        if col not in pivoted.columns:
            raise ValueError(f"[{case_name}] Expected metric column '{col}' missing after pivot.")
        pivoted[col] = pd.to_numeric(pivoted[col], errors='coerce').astype(np.float32)

    pivoted.sort_values(['cmdb_id', 'timestamp'], inplace=True)
    pivoted.reset_index(drop=True, inplace=True)
    return pivoted

case_dfs = {}
for case_name, case_dir in case_dirs.items():
    print(f"Loading {case_name}...")
    case_dfs[case_name] = load_case(case_dir, case_name)
    n = case_dfs[case_name]
    print(f"  {n.shape[0]:,} rows x {n.shape[1]} columns, {n['cmdb_id'].nunique()} containers")


## Step 4: Plot Raw Metric Patterns — One Figure Per Case

In [ ]:
import matplotlib.pyplot as plt

save_dir = Path('/kaggle/working') if IN_KAGGLE else Path('.')

for case_name, df in case_dfs.items():
    sample_containers = df['cmdb_id'].unique()[:5]
    fig, axes = plt.subplots(len(TARGET_METRICS), 1, figsize=(14, 11), sharex=False)
    for i, (col, name) in enumerate(zip(TARGET_METRICS, TARGET_NAMES)):
        for cid in sample_containers:
            cdf = df[df['cmdb_id'] == cid].sort_values('timestamp')
            axes[i].plot(cdf['timestamp'].values, cdf[col].values, label=str(cid), alpha=0.8, linewidth=0.8)
        axes[i].set_title(f'{name} -- {case_name} (raw units, full timeline)')
        axes[i].set_ylabel(name)
        axes[i].legend(fontsize=7, loc='upper right')
        axes[i].grid(alpha=0.3)
    axes[-1].set_xlabel('timestamp (s)')
    plt.tight_layout()
    save_path = save_dir / f'spike_pattern_{case_name}.png'
    plt.savefig(save_path, dpi=100)
    plt.show()
    print(f"Saved {save_path}  (containers plotted: {list(sample_containers)})")


## Step 5: Quantify Spike Presence — All 4 Cases, Ranked

In [ ]:
rows = []
for case_name, df in case_dfs.items():
    for col, name in zip(TARGET_METRICS, TARGET_NAMES):
        all_steps = []
        for cid, g in df.groupby('cmdb_id'):
            vals = g.sort_values('timestamp')[col].values
            if len(vals) > 1:
                all_steps.append(np.abs(np.diff(vals)))
        all_steps = np.concatenate(all_steps)
        mean, p95, p99, mx = all_steps.mean(), np.percentile(all_steps, 95), np.percentile(all_steps, 99), all_steps.max()
        ratio = mx / p95 if p95 > 0 else float('nan')
        rows.append({'case': case_name, 'target': name, 'mean_step': mean, 'p95_step': p95,
                     'p99_step': p99, 'max_step': mx, 'max_over_p95': ratio})

stats_df = pd.DataFrame(rows)
print(f"{'Case':<16} {'Target':<18} {'mean step':>12} {'p95 step':>12} {'max step':>12} {'max/p95':>10}")
print("-" * 84)
for case_name in VALID_CASE_NAMES:
    sub = stats_df[stats_df['case'] == case_name]
    for _, r in sub.iterrows():
        print(f"{r['case']:<16} {r['target']:<18} {r['mean_step']:>12.2f} {r['p95_step']:>12.2f} "
              f"{r['max_step']:>12.2f} {r['max_over_p95']:>9.1f}x")
    print()

print("="*70)
print("RANKED BY max/p95 RATIO (highest = most naturally spiky)")
print("="*70)
ranked = stats_df.sort_values('max_over_p95', ascending=False)
print(ranked[['case', 'target', 'max_over_p95']].head(10).to_string(index=False))

print()
print("A large max/p95 ratio (e.g. > 5-10x) means rare big jumps exist naturally for that")
print("case/target combination. If one case ranks consistently higher across multiple targets,")
print("that case is a stronger candidate for a natural-burst test set than injecting synthetic")
print("spikes into complex_case1 -- it reflects real workload behavior, not an artificial one.")
